# Scam Classification Pipeline Reproducibility Notebook
This notebook demonstrates the end-to-end process of pulling raw data from DagsHub, compiling the Universal Stratified Dataset, training two sequence classification models (DistilBERT and ModernBERT), and evaluating their accuracy on a 4,000-row held-out test set.

## 1. Setup Environment
First, we install the necessary libraries. We disable MLflow tracking for this run since this is just a local reproducibility test.

In [ ]:
!pip install transformers datasets pandas scikit-learn dagshub torch matplotlib seaborn

import os
os.environ['DISABLE_MLFLOW_INTEGRATION'] = 'True'

## 2. Authenticate with DagsHub
We need to authenticate to pull the raw datasets from the DagsHub S3 bucket.

In [ ]:
import dagshub
from google.colab import userdata # if on Colab

# Ensure you have your DAGSHUB_TOKEN set
DAGSHUB_TOKEN = 'YOUR_TOKEN_HERE'
os.environ['DAGSHUB_USER'] = 'YOUR_USERNAME'
os.environ['DAGSHUB_TOKEN'] = DAGSHUB_TOKEN

repo_owner = 'tanu320' # Update if forked
repo_name = '2026SU_MS_DSP_422-DL_SEC61_Machine_Learning_Spam_detection'
repo_id = f'{repo_owner}/{repo_name}'
s3_client = dagshub.get_repo_bucket_client(repo_id)

## 3. Data Preparation & Curation
We download three distinct datasets (LLM Synthetic, Teeconnie, Legacy) and stratify them into a perfectly balanced 2,550-row training set and a 4,000-row test set. All rows are scrubbed of formatting leakage.

In [ ]:
import pandas as pd
import json, re, zipfile, glob, random

def clean_text(text):
    text = str(text)
    text = re.sub(r'(?i)(innocent|suspect):\s*', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

print("1. Loading LLM Data...")
s3_client.download_file(repo_name, 'data/raw_jsons/scam_call_hard_examples_250.json', 'hard.json')
s3_client.download_file(repo_name, 'data/raw_jsons/scam_call_transcripts_250_combined.json', 'comb.json')
synth_data = json.load(open('hard.json')) + json.load(open('comb.json'))
df_synth = pd.DataFrame(synth_data)
df_synth['text'] = df_synth['text'].apply(clean_text)

print("2. Loading Teeconnie Data...")
s3_client.download_file(repo_name, 'data/raw_teeconnie/teeconnie_dataset.zip', 'teeconnie.zip')
with zipfile.ZipFile('teeconnie.zip', 'r') as zipf: zipf.extractall("teeconnie_raw/")
txt_files = glob.glob("teeconnie_raw/**/*non*scam*.txt", recursive=True)
content = open(txt_files[0], "r", encoding="utf-8", errors="ignore").read()
entries = [e.strip() for e in content.split("\n\n") if e.strip()]
random.seed(42)
random.shuffle(entries)
df_tee = pd.DataFrame({"text": entries, "label": [0]*len(entries)})
df_tee['text'] = df_tee['text'].apply(clean_text)

print("3. Loading Legacy Data...")
s3_client.download_file(repo_name, 'data/legacy_composite/composite_train.csv', 'l_train.csv')
s3_client.download_file(repo_name, 'data/legacy_composite/composite_test.csv', 'l_test.csv')
df_leg = pd.concat([pd.read_csv('l_train.csv'), pd.read_csv('l_test.csv')]).dropna(subset=['text', 'label'])
df_leg['text'] = df_leg['text'].apply(clean_text)
leg_scam = df_leg[df_leg['label'] == 1].sample(frac=1, random_state=42)
leg_legit = df_leg[df_leg['label'] == 0].sample(frac=1, random_state=42)

print("4. Constructing Sets...")
train_df = pd.concat([
    df_synth[df_synth['label']==1], df_synth[df_synth['label']==0],
    df_tee.iloc[:350], leg_scam.iloc[:850], leg_legit.iloc[:850]
]).sample(frac=1, random_state=42).reset_index(drop=True)

test_df = pd.concat([
    leg_scam.iloc[850:2850], leg_legit.iloc[850:1850], df_tee.iloc[350:1350]
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")

## 4. Helper Function for Plotting
This block defines a function to plot the Confusion Matrix and a Bar Chart of the summary statistics for a given model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_evaluation_results(model_name, metrics, trainer, eval_dataset, true_labels):
    # Print Metrics
    print(f"\n--- Final {model_name} Metrics ---")
    for k, v in metrics.items():
        print(f"{k}: {v}")
        
    # Get predictions for confusion matrix
    print(f"\nGenerating predictions for {model_name}...")
    preds_output = trainer.predict(eval_dataset)
    y_pred = np.argmax(preds_output.predictions, axis=-1)
    cm = confusion_matrix(true_labels, y_pred)
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['Legit', 'Scam'], yticklabels=['Legit', 'Scam'])
    axes[0].set_title(f'{model_name} Confusion Matrix')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    
    # Bar Chart
    stat_keys = ['eval_accuracy', 'eval_f1', 'eval_precision', 'eval_recall']
    stat_vals = [metrics.get(k, 0) for k in stat_keys]
    sns.barplot(x=['Accuracy', 'F1 Score', 'Precision', 'Recall'], y=stat_vals, ax=axes[1], palette='viridis')
    axes[1].set_ylim(0.8, 1.0)
    axes[1].set_title(f'{model_name} Summary Statistics')
    
    plt.tight_layout()
    plt.show()

## 5. Model Training: DistilBERT
We first train the DistilBERT model (512 token limit).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate
import numpy as np

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)
true_labels = test_df['label'].tolist()

distil_name = "distilbert-base-uncased"
distil_tokenizer = AutoTokenizer.from_pretrained(distil_name)

def tokenize_distil(examples):
    return distil_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

distil_train_tok = train_ds.map(tokenize_distil, batched=True)
distil_test_tok = test_ds.map(tokenize_distil, batched=True)

distil_model = AutoModelForSequenceClassification.from_pretrained(distil_name, num_labels=2)

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return clf_metrics.compute(predictions=predictions, references=labels)

distil_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none" 
)

distil_trainer = Trainer(
    model=distil_model,
    args=distil_args,
    train_dataset=distil_train_tok,
    eval_dataset=distil_test_tok,
    compute_metrics=compute_metrics
)

print("Starting DistilBERT Training...")
distil_trainer.train()

print("\nRunning Final DistilBERT Evaluation...")
distil_metrics = distil_trainer.evaluate(eval_dataset=distil_test_tok)
plot_evaluation_results('DistilBERT', distil_metrics, distil_trainer, distil_test_tok, true_labels)

## 6. Model Training: ModernBERT (8192 Token Context)
Now we train ModernBERT. Because the context limit is 8192 tokens, we must use Gradient Checkpointing and Gradient Accumulation (batch size 1) to prevent CUDA Out of Memory errors on standard GPUs.

In [ ]:
modern_name = "answerdotai/ModernBERT-base"
modern_tokenizer = AutoTokenizer.from_pretrained(modern_name)

def tokenize_modern(examples):
    return modern_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=8192)

modern_train_tok = train_ds.map(tokenize_modern, batched=True)
modern_test_tok = test_ds.map(tokenize_modern, batched=True)

modern_model = AutoModelForSequenceClassification.from_pretrained(modern_name, num_labels=2)

modern_args = TrainingArguments(
    output_dir="./modernbert_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=1, # Physical batch size 1 to prevent OOM
    gradient_accumulation_steps=16, # Simulate batch size 16
    gradient_checkpointing=True, # Crucial for 8192 tokens on 15GB VRAM GPUs
    num_train_epochs=3,
    report_to="none"
)

modern_trainer = Trainer(
    model=modern_model,
    args=modern_args,
    train_dataset=modern_train_tok,
    eval_dataset=modern_test_tok,
    compute_metrics=compute_metrics
)

print("Starting ModernBERT Training...")
modern_trainer.train()

print("\nRunning Final ModernBERT Evaluation...")
modern_metrics = modern_trainer.evaluate(eval_dataset=modern_test_tok)
plot_evaluation_results('ModernBERT', modern_metrics, modern_trainer, modern_test_tok, true_labels)